# Testing and Evaluating Gemini Functions

This notebook covers three things:

1. **Two Gemini-backed functions**
   - `classify_question` — sorts a user question into one of: Employment, General Information, Emergency Services, or Tax Related.
   - `generate_social_post` — writes a short social media post for a government announcement (weather emergencies, holidays, school closings, etc.).
2. **Unit tests** for each function using pytest, with the Gemini model mocked so the tests are fast and deterministic.
3. **Evaluation** using the Vertex AI Gen AI Evaluation API to compare responses produced by different prompts.

Both functions split a pure prompt-builder (testable without the API) from the model call, and the model is injectable so tests can pass a mock.

## Setup

In [1]:
!pip install -q "google-cloud-aiplatform[evaluation]" pytest pandas

## Configuration

The evaluation service runs in a region, so we use `us-central1`. We use `gemini-2.5-flash` (the 2.0 models were retired in June 2026).

In [2]:
PROJECT_ID = "qwiklabs-gcp-00-fc2622edeeb6"   # update to match your lab project
LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Vertex AI initialized in", LOCATION)

Vertex AI initialized in us-central1


## The functions

Written to a module so pytest can import them. Each function builds a prompt, sends it to Gemini, and normalizes the result. The `model` argument defaults to a shared `GenerativeModel` but can be overridden with a mock in tests.

Two practical details for Gemini 2.5: it is a *thinking* model that spends tokens reasoning before answering, so we give a generous `max_output_tokens` budget. We also guard the `.text` access — if a response comes back empty (blocked, or all budget consumed), `_generate` returns an empty string and the caller's fallback handles it instead of crashing.

In [3]:
%%writefile gov_assistant.py
"""Gemini-backed functions for a local government assistant.

Two tasks:
  - classify_question: route a question to a department category.
  - generate_social_post: draft a social media post for an announcement.

Each task separates prompt construction from the model call, and the model is
injectable so unit tests can pass a mock instead of calling the live API.
"""

from vertexai.generative_models import GenerativeModel

DEFAULT_MODEL_NAME = "gemini-2.5-flash"
_cached_model = None


def get_model(model_name=DEFAULT_MODEL_NAME):
    """Lazily create and reuse a GenerativeModel instance."""
    global _cached_model
    if _cached_model is None:
        _cached_model = GenerativeModel(model_name)
    return _cached_model


def _generate(prompt, model=None, temperature=0.2, max_output_tokens=2048):
    """Send a prompt to Gemini and return the stripped response text.

    Gemini 2.5 is a thinking model, so the token budget needs headroom for
    both reasoning and the answer. If the response has no text (blocked or
    empty), return an empty string and let the caller fall back.
    """
    model = model or get_model()
    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_output_tokens,
        },
    )
    try:
        return response.text.strip()
    except (ValueError, AttributeError):
        return ""


# --- Function 1: question classification ---

CATEGORIES = ("Employment", "General Information", "Emergency Services", "Tax Related")


def build_classification_prompt(question):
    """Build the prompt that asks Gemini to pick one category."""
    if not question or not question.strip():
        raise ValueError("question must be a non-empty string")
    category_list = ", ".join(CATEGORIES)
    return (
        "You are a router for a local government help line. "
        "Classify the user's question into exactly one of these categories: "
        f"{category_list}.\n"
        "Respond with only the category name and nothing else.\n\n"
        f"Question: {question}\nCategory:"
    )


def classify_question(question, model=None):
    """Classify a question into one of CATEGORIES.

    The raw model output is matched against the known categories
    (case-insensitive). Unrecognized or empty output falls back to
    'General Information'.
    """
    prompt = build_classification_prompt(question)
    raw = _generate(prompt, model=model, temperature=0.0).lower()
    for category in CATEGORIES:
        if category.lower() in raw:
            return category
    return "General Information"


# --- Function 2: social media post generation ---

def build_social_post_prompt(announcement, platform="Twitter"):
    """Build the prompt for drafting a social media post."""
    if not announcement or not announcement.strip():
        raise ValueError("announcement must be a non-empty string")
    return (
        f"Write a short, clear {platform} post for a local government account "
        "announcing the following. Use a calm, informative tone, include a "
        "relevant hashtag, and keep it under 280 characters.\n\n"
        f"Announcement: {announcement}\nPost:"
    )


def generate_social_post(announcement, platform="Twitter", model=None):
    """Generate a social media post for a government announcement."""
    prompt = build_social_post_prompt(announcement, platform=platform)
    return _generate(prompt, model=model, temperature=0.4, max_output_tokens=1024)


Overwriting gov_assistant.py


## Quick manual check

Live calls to confirm both functions work before testing and evaluating them.

In [4]:
import gov_assistant

print("Classification examples:")
print(" ", gov_assistant.classify_question("How do I apply for a job with the city?"))
print(" ", gov_assistant.classify_question("There's a gas leak on Main Street, who do I call?"))
print(" ", gov_assistant.classify_question("When are my property taxes due?"))
print(" ", gov_assistant.classify_question("What are the library's opening hours?"))

print("\nSocial post example:")
print(gov_assistant.generate_social_post(
    "All public schools will be closed tomorrow due to a severe winter storm warning."
))

Classification examples:


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


  Employment
  Emergency Services
  Tax Related
  General Information

Social post example:
Please note: All public schools will be closed tomorrow due to a severe winter storm warning. Stay safe, everyone. #SchoolClosures #CommunityAlert


## Unit tests

The Gemini model is replaced with a `MagicMock`, so the tests never call the API. They cover prompt construction, category normalization and fallback, input validation, and the post generator passing the announcement through to the model.

In [5]:
%%writefile test_gov_assistant.py
"""Unit tests for gov_assistant. The Gemini model is mocked throughout."""

import pytest
from unittest.mock import MagicMock

import gov_assistant as ga


def make_model(text):
    """A fake model whose generate_content returns an object with .text."""
    model = MagicMock()
    model.generate_content.return_value = MagicMock(text=text)
    return model


# --- classify_question ---

def test_classification_prompt_lists_all_categories():
    prompt = ga.build_classification_prompt("Where do I pay taxes?")
    for category in ga.CATEGORIES:
        assert category in prompt
    assert "Where do I pay taxes?" in prompt


@pytest.mark.parametrize("raw,expected", [
    ("Employment", "Employment"),
    ("employment", "Employment"),
    ("Tax Related", "Tax Related"),
    ("This is Emergency Services.", "Emergency Services"),
    ("General Information", "General Information"),
    ("banana", "General Information"),   # unrecognized -> fallback
    ("", "General Information"),          # empty -> fallback
])
def test_classify_question_normalizes(raw, expected):
    model = make_model(raw)
    assert ga.classify_question("some question", model=model) == expected


def test_classify_question_returns_known_category():
    model = make_model("Tax Related")
    assert ga.classify_question("property taxes?", model=model) in ga.CATEGORIES


@pytest.mark.parametrize("bad", ["", "   ", None])
def test_classification_rejects_empty(bad):
    with pytest.raises(ValueError):
        ga.build_classification_prompt(bad)


# --- generate_social_post ---

def test_social_prompt_includes_announcement_and_platform():
    prompt = ga.build_social_post_prompt("Roads closed downtown", platform="Facebook")
    assert "Roads closed downtown" in prompt
    assert "Facebook" in prompt


def test_generate_social_post_strips_and_returns_text():
    model = make_model("  Schools closed tomorrow. Stay safe! #SnowDay  ")
    result = ga.generate_social_post("Schools closed tomorrow", model=model)
    assert result == "Schools closed tomorrow. Stay safe! #SnowDay"
    model.generate_content.assert_called_once()


def test_generate_social_post_passes_announcement_to_model():
    model = make_model("ok")
    ga.generate_social_post("Fourth of July parade at noon", model=model)
    sent_prompt = model.generate_content.call_args[0][0]
    assert "Fourth of July parade at noon" in sent_prompt


@pytest.mark.parametrize("bad", ["", "   ", None])
def test_social_post_rejects_empty(bad):
    with pytest.raises(ValueError):
        ga.build_social_post_prompt(bad)


Overwriting test_gov_assistant.py


Run the tests:

In [6]:
!python -m pytest test_gov_assistant.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 18 items                                                             

test_gov_assistant.py::test_classification_prompt_lists_all_categories PASSED [  5%]
test_gov_assistant.py::test_classify_question_normalizes[Employment-Employment] PASSED [ 11%]
test_gov_assistant.py::test_classify_question_normalizes[employment-Employment] PASSED [ 16%]
test_gov_assistant.py::test_classify_question_normalizes[Tax Related-Tax Related] PASSED [ 22%]
test_gov_assistant.py::test_classify_question_normalizes[This is Emergency Services.-Emergency Services] PASSED [ 27%]
test_gov_assistant.py::test_classify_question_normalizes[General Information-General Information] PASSED [ 33%]
test_gov_assistant.py::test_classify_question_normalizes[ba

## Evaluation with the Gen AI Evaluation API

The requirement is to compare Gemini responses from **different prompts**. For the social post task we define two prompt styles and see which the evaluation judge prefers.

First, list the available example metrics.

In [7]:
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples, PointwiseMetric
import pandas as pd

print(MetricPromptTemplateExamples.list_example_metric_names())

['coherence', 'fluency', 'safety', 'groundedness', 'instruction_following', 'verbosity', 'text_quality', 'summarization_quality', 'question_answering_quality', 'multi_turn_chat_quality', 'multi_turn_safety', 'pairwise_coherence', 'pairwise_fluency', 'pairwise_safety', 'pairwise_groundedness', 'pairwise_instruction_following', 'pairwise_verbosity', 'pairwise_text_quality', 'pairwise_summarization_quality', 'pairwise_question_answering_quality', 'pairwise_multi_turn_chat_quality', 'pairwise_multi_turn_safety']


### Two prompt variants

Both take an announcement and produce a post, but with different instructions: a plain factual style versus a more engaging, community-friendly style.

In [8]:
def post_prompt_plain(announcement):
    return (
        "Write a short, factual government social media post for this "
        "announcement. Keep it under 280 characters.\n\n"
        f"Announcement: {announcement}\nPost:"
    )

def post_prompt_engaging(announcement):
    return (
        "Write a warm, community-friendly government social media post for this "
        "announcement. Be reassuring, add a relevant emoji and hashtag, and keep "
        "it under 280 characters.\n\n"
        f"Announcement: {announcement}\nPost:"
    )

announcements = [
    "All public schools closed tomorrow due to a severe winter storm.",
    "City offices will be closed Monday for the Independence Day holiday.",
    "A boil-water advisory is in effect for the north side until further notice.",
    "Free flu shots available at the community center this Saturday.",
    "Trash collection is delayed one day this week due to the holiday.",
]

def generate(prompt_fn):
    return [gov_assistant._generate(prompt_fn(a), temperature=0.4, max_output_tokens=1024)
            for a in announcements]

eval_df = pd.DataFrame({
    "prompt": announcements,
    "plain": generate(post_prompt_plain),
    "engaging": generate(post_prompt_engaging),
})
eval_df

,prompt,plain,engaging
0,All public schools closed tomorrow due to a se...,ALERT: All public schools closed tomorrow due ...,"Heads up, community! Due to the severe winter ..."
1,City offices will be closed Monday for the Ind...,"City offices will be closed Monday, July 3rd, ...",Wishing our community a wonderful Independence...
2,A boil-water advisory is in effect for the nor...,North Side: Boil-water advisory in effect unti...,"North side neighbors, a boil-water advisory is..."
3,Free flu shots available at the community cent...,Free flu shots available this Saturday at the ...,Hi neighbors! 👋 Get your free flu shot this Sa...
4,Trash collection is delayed one day this week ...,Holiday Alert: Trash collection is delayed one...,"Heads up, neighbors! 🗓️ Due to the holiday, tr..."


### Pointwise evaluation

Score each prompt variant's posts on fluency and verbosity. Bring-your-own-response: the dataset supplies the `response` column.

In [9]:
pointwise_metrics = [
    MetricPromptTemplateExamples.Pointwise.FLUENCY,
    MetricPromptTemplateExamples.Pointwise.VERBOSITY,
]

def run_pointwise(response_col):
    df = pd.DataFrame({"prompt": eval_df["prompt"], "response": eval_df[response_col]})
    return EvalTask(dataset=df, metrics=pointwise_metrics).evaluate()

plain_result = run_pointwise("plain")
engaging_result = run_pointwise("engaging")

print("Plain prompt:   ", plain_result.summary_metrics)
print("Engaging prompt:", engaging_result.summary_metrics)

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 10 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 10/10 [00:22<00:00,  2.23s/it]
INFO:vertexai.evaluation._evaluation:All 10 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:22.326895862999663 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 10 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 10/10 [00:46<00:00,  4.66s/it]
INFO:vertexai.evaluation._evaluation:All 10 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:46.62820203200317 seconds


Plain prompt:    {'row_count': 5, 'fluency/mean': np.float64(5.0), 'fluency/std': 0.0, 'verbosity/mean': np.float64(0.0), 'verbosity/std': 0.0}
Engaging prompt: {'row_count': 5, 'fluency/mean': np.float64(4.8), 'fluency/std': 0.4472135954999579, 'verbosity/mean': np.float64(0.6), 'verbosity/std': 0.5477225575051662}


Row-level scores and judge rationale for the engaging variant:

In [10]:
engaging_result.metrics_table

,prompt,response,fluency/explanation,fluency/score,verbosity/explanation,verbosity/score
0,All public schools closed tomorrow due to a se...,"Heads up, community! Due to the severe winter ...","The response is free of grammatical errors, de...",5.0,The response reiterates the core information f...,1.0
1,City offices will be closed Monday for the Ind...,Wishing our community a wonderful Independence...,The response is completely free of grammatical...,5.0,The response includes the core information fro...,1.0
2,A boil-water advisory is in effect for the nor...,"North side neighbors, a boil-water advisory is...","The response is mostly fluent, with clear word...",4.0,"The response is perfectly concise, taking the ...",0.0
3,Free flu shots available at the community cent...,Hi neighbors! 👋 Get your free flu shot this Sa...,The response is completely free of grammatical...,5.0,The AI response is somewhat verbose as it expa...,1.0
4,Trash collection is delayed one day this week ...,"Heads up, neighbors! 🗓️ Due to the holiday, tr...","The response is free of grammatical errors, us...",5.0,The response is perfectly concise. It rephrase...,0.0


### Pairwise comparison

Put the two prompts head-to-head. The candidate is the engaging variant (`response`); the baseline is the plain variant (`baseline_model_response`). The judge picks a winner per row.

In [11]:
pairwise_df = pd.DataFrame({
    "prompt": eval_df["prompt"],
    "response": eval_df["engaging"],
    "baseline_model_response": eval_df["plain"],
})

pairwise_result = EvalTask(
    dataset=pairwise_df,
    metrics=[MetricPromptTemplateExamples.Pairwise.INSTRUCTION_FOLLOWING],
).evaluate()

print(pairwise_result.summary_metrics)
pairwise_result.metrics_table

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 5 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 5/5 [00:22<00:00,  4.60s/it]
INFO:vertexai.evaluation._evaluation:All 5 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:22.99063260299954 seconds


{'row_count': 5, 'pairwise_instruction_following/candidate_model_win_rate': np.float64(0.4), 'pairwise_instruction_following/baseline_model_win_rate': np.float64(0.2)}


,prompt,response,baseline_model_response,pairwise_instruction_following/explanation,pairwise_instruction_following/pairwise_choice
0,All public schools closed tomorrow due to a se...,"Heads up, community! Due to the severe winter ...",ALERT: All public schools closed tomorrow due ...,Both responses effectively convey the core inf...,TIE
1,City offices will be closed Monday for the Ind...,Wishing our community a wonderful Independence...,"City offices will be closed Monday, July 3rd, ...",CANDIDATE response accurately reflects a typic...,CANDIDATE
2,A boil-water advisory is in effect for the nor...,"North side neighbors, a boil-water advisory is...",North Side: Boil-water advisory in effect unti...,BASELINE response directly conveys the advisor...,BASELINE
3,Free flu shots available at the community cent...,Hi neighbors! 👋 Get your free flu shot this Sa...,Free flu shots available this Saturday at the ...,CANDIDATE response is slightly more engaging a...,CANDIDATE
4,Trash collection is delayed one day this week ...,"Heads up, neighbors! 🗓️ Due to the holiday, tr...",Holiday Alert: Trash collection is delayed one...,Both responses effectively transform the given...,TIE


## Summary

- Built two Gemini functions: `classify_question` (Employment / General Information / Emergency Services / Tax Related) and `generate_social_post` for government announcements. Both separate prompt-building from the model call for testability.
- Wrote pytest unit tests for each, with the model mocked, covering prompt construction, category normalization and fallback, output stripping, and input validation.
- Used the Vertex AI Gen AI Evaluation API to compare two different prompt styles for the social post task, both pointwise (fluency, verbosity) and pairwise (instruction following).

Notes:
- Uses `gemini-2.5-flash`; the 2.0 Flash models were retired in June 2026.
- If a metric enum name errors, check the printed `list_example_metric_names()` output and adjust.
- If `evaluate()` reports a missing staging bucket, re-run `vertexai.init` with `staging_bucket="gs://your-bucket"`.